# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [5]:
%pip install -q langchain langchain-openai langchain-anthropic pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [1]:
# ============================================================
# CONFIGURATION - Edit these values
# ============================================================
import os
from dotenv import load_dotenv

load_dotenv(".env")

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Model settings
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Email settings (Gmail with App Password)
EMAIL_SENDER = "provide your email"
EMAIL_PASSWORD = "password"

# Meal generation settings
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ["salt", "pepper", "oil", "garlic", "water"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [4]:
if not OPENAI_API_KEY:
    raise ValueError(
        "Configuration | Missing OPENAI_API_KEY. "
        "Set it in the .env file or environment variables."
    )

# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [5]:
saved_key = OPENAI_API_KEY

try:
    OPENAI_API_KEY = None

    if not OPENAI_API_KEY:
        raise ValueError(
            "Configuration | Missing OPENAI_API_KEY. "
            "Set it in the .env file or environment variables."
        )
except ValueError as e:
    print(e)
finally:
    OPENAI_API_KEY = saved_key

Configuration | Missing OPENAI_API_KEY. Set it in the .env file or environment variables.


## 3. Pydantic Models for Validation

In [6]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

## 4. LangChain Setup & Prompts

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Get the configured LLM."""
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

## 5. Meal Generation Logic

In [8]:
import json
import time
from pydantic import ValidationError
from openai import APIConnectionError, APIStatusError, APIError
from unittest.mock import patch
from types import SimpleNamespace

def clean_json_response(content):
    """Extract JSON from model response."""
    if "```json" in content:
        content = content.split("```json")[1].split("```")[0]
    elif "```" in content:
        content = content.split("```")[1].split("```")[0]
    return content.strip()  

def parse_and_validate_meal_plan(content):
    meal_plan_data = json.loads(content)
    meal_plan = MealPlan(**meal_plan_data)
    return meal_plan

class MealPlanGenerator:
    """Generates weekly themed meal plans."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)
        self.shopping_parser = JsonOutputParser(pydantic_object=ShoppingItem)

    def request_meal_plan(
        self,
        dietary_preference,
        theme_requests,
        additional_preferences
    ):
        recipe_chain = MEAL_PLAN_PROMPT | self.llm

        return recipe_chain.invoke({
            "dietary_preference": dietary_preference,
            "theme_requests": theme_requests if theme_requests else "random diverse cuisines",
            "additional_preferences": additional_preferences if additional_preferences else "none",
            "format_instructions": self.recipe_parser.get_format_instructions()
        })

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none"
    ) -> dict:
        """Generate a complete meal plan with shopping list."""
        start_time = time.time()

        try:
            result = self.request_meal_plan(
                dietary_preference,
                theme_requests,
                additional_preferences
            )
        
            content = clean_json_response(result.content)
            meal_plan = parse_and_validate_meal_plan(content)
            generation_time = time.time() - start_time

            return {
                "success": True,
                "meal_plan": meal_plan.model_dump(),
                "generation_time": round(generation_time, 2),
                "error": None
            }

        except json.JSONDecodeError as e:
            return {"success": False, "meal_plan": None, "error": f"MealPlanGenerator.generate | JSONDecodeError | Line {e.lineno}, column {e.colno}: {e.msg}. Check the model response is valid JSON."}
        except ValidationError as e:
            return {"success": False,"meal_plan": None,"error": f"MealPlanGenerator.generate | ValidationError | {e}. Check the required fields and recipe rules."}
        except APIConnectionError as e:
            return {"success": False,"meal_plan": None,"error": f"MealPlanGenerator.request_meal_plan | {type(e).__name__}: {e}. Check your connection and API timeout settings."}
        except APIStatusError as e:
            return {"success": False,"meal_plan": None,"error": f"MealPlanGenerator.request_meal_plan | {type(e).__name__} | HTTP {e.status_code}: {e}. Check API permissions, request parameters, and quota."}
        except APIError as e:
            return {"success": False,"meal_plan": None,"error": f"MealPlanGenerator.request_meal_plan | {type(e).__name__}: {e}. Review the API error details and request configuration."}
        except Exception as e:
            return {"success": False, "meal_plan": None, "error": f"MealPlanGenerator.generate | {type(e).__name__}: {e}. Check the failing operation and its configuration."}


# Initialize generator
generator = MealPlanGenerator()

In [9]:
with patch.object(
    type(generator.llm),
    "invoke",
    return_value=SimpleNamespace(content='{}')
):
    validation_test = generator.generate("omnivore")

print(validation_test)

{'success': False, 'meal_plan': None, 'error': 'MealPlanGenerator.generate | ValidationError | 3 validation errors for MealPlan\ndietary_preference\n  Field required [type=missing, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nrecipes\n  Field required [type=missing, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nshopping_list\n  Field required [type=missing, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing. Check the required fields and recipe rules.'}


In [10]:
with patch.object(
    type(generator.llm),
    "invoke",
    return_value=SimpleNamespace(content='{"recipes":')
):
    test_result = generator.generate("omnivore")

print(test_result)

{'success': False, 'meal_plan': None, 'error': 'MealPlanGenerator.generate | JSONDecodeError | Line 1, column 12: Expecting value. Check the model response is valid JSON.'}


In [11]:
with patch.object(
    type(generator.llm),
    "invoke",
    side_effect=RuntimeError("Simulated call failure")
):
    call_test = generator.generate("omnivore")

print(call_test)

{'success': False, 'meal_plan': None, 'error': 'MealPlanGenerator.generate | RuntimeError: Simulated call failure. Check the failing operation and its configuration.'}


In [12]:
assert clean_json_response('{"test": 1}') == '{"test": 1}'
assert clean_json_response('```json\n{"test": 1}\n```') == '{"test": 1}'
assert clean_json_response('```\n{"test": 1}\n```') == '{"test": 1}'

print("3 cleaning tests passed")

3 cleaning tests passed


In [13]:
import httpx

connection_error = APIConnectionError(
    message="Simulated connection failure",
    request=httpx.Request("POST", "https://api.openai.com/v1/chat/completions")
)

with patch.object(
    type(generator.llm),
    "invoke",
    side_effect=connection_error
):
    connection_test = generator.generate("omnivore")

print(connection_test)

{'success': False, 'meal_plan': None, 'error': 'MealPlanGenerator.request_meal_plan | APIConnectionError: Simulated connection failure. Check your connection and API timeout settings.'}


In [14]:
request = httpx.Request(
    "POST",
    "https://api.openai.com/v1/chat/completions"
)

response = httpx.Response(403, request=request)

status_error = APIStatusError(
    "Simulated permission error",
    response=response,
    body={"error": {"message": "Simulated permission error"}}
)

with patch.object(
    type(generator.llm),
    "invoke",
    side_effect=status_error
):
    status_test = generator.generate("omnivore")

print(status_test)

{'success': False, 'meal_plan': None, 'error': 'MealPlanGenerator.request_meal_plan | APIStatusError | HTTP 403: Simulated permission error. Check API permissions, request parameters, and quota.'}


In [15]:
api_error = APIError(
    "Simulated API error",
    request=httpx.Request(
        "POST",
        "https://api.openai.com/v1/chat/completions"
    ),
    body=None
)

with patch.object(
    type(generator.llm),
    "invoke",
    side_effect=api_error
):
    api_test = generator.generate("omnivore")

print(api_test)

{'success': False, 'meal_plan': None, 'error': 'MealPlanGenerator.request_meal_plan | APIError: Simulated API error. Review the API error details and request configuration.'}


## 6. Email Functionality

In [16]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart


def format_meal_plan_email(meal_plan: dict) -> str:
    """Format meal plan as readable email content (shopping list first)."""
    lines = []
    lines.append("=" * 40)
    lines.append(f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})")
    lines.append("=" * 40)
    lines.append("")

    # Shopping list FIRST
    lines.append("SHOPPING LIST")
    lines.append("-" * 20)
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines.append("")

    # Recipes
    lines.append("=" * 40)
    lines.append("RECIPES")
    lines.append("=" * 40)
    for recipe in meal_plan['recipes']:
        lines.append(f"\n{recipe['day'].upper()} - {recipe['theme']}")
        lines.append(f"{recipe['dish_name']}")
        ingredients = ", ".join([f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients']])
        lines.append(f"Ingredients: {ingredients}")
        lines.append(f"Instructions: {recipe['instructions']}")

    return "\n".join(lines)


def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Send meal plan via email."""
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_SENDER
        msg['To'] = recipient_email
        msg['Subject'] = "Your Weekly Meal Plan"

        body = format_meal_plan_email(meal_plan)
        msg.attach(MIMEText(body, 'plain'))

        with smtplib.SMTP('smtp.gmail.com', 587) as server:
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.send_message(msg)

        return {"success": True, "message": f"Email sent to {recipient_email}"}
    except Exception as e:
        return {"success": False, "message": f"Failed to send email: {str(e)}"}

## 7. Validation

In [17]:
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Run quick programmatic validation checks (not LLM-based)."""
    if not meal_plan_result.get("success"):
        return {"passed": False, "errors": [meal_plan_result.get("error")]}

    meal_plan = meal_plan_result["meal_plan"]
    errors = []

    # Check 1: 7 recipes
    if len(meal_plan["recipes"]) != 7:
        errors.append(f"Expected 7 recipes, got {len(meal_plan['recipes'])}")

    # Check 2: Unique themes
    themes = [r["theme"].lower() for r in meal_plan["recipes"]]
    if len(themes) != len(set(themes)):
        duplicates = [t for t in themes if themes.count(t) > 1]
        errors.append(f"Duplicate themes: {set(duplicates)}")

    # Check 3: 5 ingredients per recipe
    for recipe in meal_plan["recipes"]:
        if len(recipe["ingredients"]) != 5:
            errors.append(f"{recipe['day']}: {len(recipe['ingredients'])} ingredients (expected 5)")

    return {"passed": len(errors) == 0, "errors": errors}

In [18]:
successful_result = generator.generate("omnivore")

print("Success:", successful_result["success"])

if successful_result["success"]:
    print("Recipes:", len(successful_result["meal_plan"]["recipes"]))
    print("Validation:", run_basic_validation(successful_result))
else:
    print(successful_result["error"])

Success: True
Recipes: 7
Validation: {'passed': True, 'errors': []}


In [21]:
print(json.dumps(successful_result, indent=2, ensure_ascii=False))

{
  "success": true,
  "meal_plan": {
    "dietary_preference": "omnivore",
    "recipes": [
      {
        "day": "Monday",
        "theme": "Italian",
        "dish_name": "Pasta Primavera",
        "ingredients": [
          {
            "name": "Pasta",
            "quantity": "200g"
          },
          {
            "name": "Bell peppers",
            "quantity": "100g"
          },
          {
            "name": "Zucchini",
            "quantity": "100g"
          },
          {
            "name": "Cherry tomatoes",
            "quantity": "150g"
          },
          {
            "name": "Parmesan cheese",
            "quantity": "50g"
          }
        ],
        "instructions": "Cook pasta according to package instructions. Sauté bell peppers, zucchini, and cherry tomatoes in a pan. Combine cooked pasta with sautéed vegetables and top with grated Parmesan cheese."
      },
      {
        "day": "Tuesday",
        "theme": "Mexican",
        "dish_name": "Chicken Fa

## 8. Gradio Interface

In [19]:
import gradio as gr
import time as pytime


def format_shopping_list(meal_plan: dict) -> str:
    """Format shopping list section."""
    lines = ["## Shopping List\n"]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format a single recipe compactly."""
    ingredients = ", ".join([f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients']])
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{ingredients}\n\n**Instructions:** {recipe['instructions']}\n"

def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Generate meal plan with streaming output."""
    output = "Generating your meal plan..."
    yield output

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=theme_requests if theme_requests.strip() else "random",
        additional_preferences=additional_prefs if additional_prefs.strip() else "none"
    )

    if not result["success"]:
        yield f"Error: {result['error']}"
        return

    meal_plan = result["meal_plan"]

    # Show shopping list first
    output = f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    output += format_shopping_list(meal_plan) + "\n\n---\n\n"
    output += "## Recipes\n\n"
    yield output
    pytime.sleep(0.1)

    # Stream recipes one by one
    for recipe in meal_plan['recipes']:
        output += format_single_recipe(recipe) + "\n"
        yield output
        pytime.sleep(0.05)

    # Validation status
    validation = run_basic_validation(result)
    status = "All checks passed" if validation["passed"] else f"Issues: {', '.join(validation['errors'])}"
    output += f"---\n*Generated in {result['generation_time']}s | {status}*"
    yield output


def send_email_ui(email, dietary_pref, theme_requests, additional_prefs):
    """Generate and send meal plan via email."""
    if not email or "@" not in email:
        return "Enter a valid email address."

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=theme_requests if theme_requests.strip() else "random",
        additional_preferences=additional_prefs if additional_prefs.strip() else "none"
    )

    if not result["success"]:
        return f"Generation failed: {result['error']}"

    email_result = send_meal_plan_email(email, result["meal_plan"])
    return email_result["message"]


# Create Gradio interface
with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=["omnivore", "vegetarian", "vegan", "keto"],
                value="omnivore",
                label="Diet"
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display]
    )

    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, dietary_dropdown, theme_input, additional_input],
        outputs=[email_status]
    )

/opt/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
recipe_test = {
    "day": "Monday",
    "theme": "Italian",
    "dish_name": "Simple Pasta",
    "ingredients": [
        {"name": "pasta", "quantity": "200g"}
    ],
    "instructions": "Cook the pasta and serve."
}

print(format_single_recipe(recipe_test))

**Monday** | Italian | *Simple Pasta*
200g pasta

**Instructions:** Cook the pasta and serve.



## 9. Launch Application

In [13]:
# Launch the Gradio app
app.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 10. Run Test Cases

Runs every test case through the generator and the basic validation checks. This makes 4 real API calls.

In [11]:
TEST_CASES = [
    {"dietary_preference": "omnivore", "theme_requests": "random", "additional_preferences": "none"},
    {"dietary_preference": "vegetarian", "theme_requests": "Italian, Asian, Mexican", "additional_preferences": "no mushrooms"},
    {"dietary_preference": "vegan", "theme_requests": "random", "additional_preferences": "high protein"},
    {"dietary_preference": "keto", "theme_requests": "Mediterranean, American", "additional_preferences": "none"}
]

for case in TEST_CASES:
    result = generator.generate(
        dietary_preference=case["dietary_preference"],
        theme_requests=case.get("theme_requests", "random"),
        additional_preferences=case.get("additional_preferences", "none")
    )
    validation = run_basic_validation(result)
    print(case["dietary_preference"], "->", "PASS" if validation["passed"] else validation["errors"])

omnivore -> PASS
vegetarian -> PASS
vegan -> PASS
keto -> PASS
